# B1 · 參數簡併與 MCMC 診斷 —— 為什麼這裡非 MCMC 不可

> 凌日的深度、時長、形狀由 `Rp/R*`、`a/R*`、`b` 的**組合**決定，單獨都不好定 → 強烈簡併。
> 這正是 Mean-Field 變分推論（假設參數獨立）會失敗、而 MCMC 能誠實捕捉的地方。

In [1]:
import sys, os, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, arviz as az
import data, transit_model as tm, inference as inf
DATA = os.path.abspath('../../data/B_astro')
d = data.prepare(DATA)
P = float(d['P']); pub = data.published_kepler10b()
bp, bf, be, bn = data.bin_fold(d['fold_phase'], d['fold_flux'], window=0.06, n_bins=80)
print(f"總點數={int(d['n_points'])}  BLS 週期={P:.6f} d（已發表 {pub['P']:.6f}）")

總點數=51768  BLS 週期=0.837487 d（已發表 0.837491）


In [2]:
ev = tm.TransitEvaluator(bp, P)   # 與 run_all 相同設定（64 walkers × 30000 步）
sampler, idata, tau = inf.run_emcee(ev, bf, be, seed=42)
fs = inf.flat_samples(sampler)
summ = az.summary(idata, var_names=inf.LABELS)
print(summ[['mean','sd','hdi_3%','hdi_97%','r_hat','ess_bulk']].to_string())

          mean     sd  hdi_3%  hdi_97%  r_hat  ess_bulk
rp       0.013  0.000   0.013    0.014   1.01    4493.0
a        4.055  0.376   3.286    4.517   1.02    4621.0
b        0.329  0.205   0.000    0.677   1.02    4426.0
t0       0.001  0.000   0.001    0.002   1.00   22846.0
q1       0.390  0.083   0.230    0.544   1.00   21894.0
q2       0.224  0.077   0.077    0.369   1.00   18857.0
f0       1.000  0.000   1.000    1.000   1.00   22688.0
log_jit -5.138  0.059  -5.251   -5.026   1.00   22930.0


## 1 · 先驗設計的四個要點（計劃書步驟 4）

1. **`Rp/R*` 正數**：半徑比不能為負——用有界/半正態先驗，不能讓高斯漏到負區。
2. **`cos i` 均勻，不是 `i` 均勻**：行星的軌道傾角在幾何上應對 $\cos i$ 均勻（撞擊參數 $b$ 均勻）——這是「均勻 ≠ 無資訊」與參數化不變性的真實應用。
3. **Kipping (2013) 臨邊昏暗**：$(q_1,q_2)\in[0,1]^2$ 映射到物理有效的 $(u_1,u_2)$ 三角形；淺凌日幾乎不約束 LD，故用恆星模型理論值當先驗。
4. **長曝光積分**（踩坑）：Kepler 長曝光 29.4 分鐘 > 凌日入/出時間，會抹平邊緣。不積分的話，擬合會用極端 LD 把底部磨圓、並把 Rp/R* 系統性壓低。

## 2 · 參數簡併（corner plot）

![corner](../figures/03_corner.png)

In [3]:
print(f"corr(Rp/R*, a/R*) = {np.corrcoef(fs[:,0], fs[:,1])[0,1]:+.2f}")
print(f"corr(Rp/R*, b)    = {np.corrcoef(fs[:,0], fs[:,2])[0,1]:+.2f}")
print(f"corr(a/R*, b)      = {np.corrcoef(fs[:,1], fs[:,2])[0,1]:+.2f}")

corr(Rp/R*, a/R*) = -0.87
corr(Rp/R*, b)    = +0.83
corr(a/R*, b)      = -0.94


> corner plot 上的 `Rp/R*`–`a/R*`–`b` 呈**彎曲、非高斯的香蕉形**強簡併。
> **這正是 Mean-Field VI 會失敗的地方**：它假設參數獨立、後驗為對角高斯，會把這根香蕉壓成一個小圓球，嚴重低估不確定性。MCMC（這裡用 emcee）能誠實地沿著香蕉採樣。

## 3 · MCMC 診斷

強簡併的香蕉形後驗，emcee 預設的 StretchMove 混合很慢——改用 **DEMove（差分演化）**大幅改善。

In [4]:
acc = np.mean(sampler.acceptance_fraction)
print(f"接受率 = {acc:.2f}")
print(f"最大自相關時間 τ = {np.nanmax(tau):.0f}，鏈長/τ = {30000/np.nanmax(tau):.0f}（>50 為收斂）")
print(f"最大 r_hat = {summ['r_hat'].max():.3f}，最小 ESS = {int(summ['ess_bulk'].min())}")

接受率 = 0.18
最大自相關時間 τ = 435，鏈長/τ = 69（>50 為收斂）
最大 r_hat = 1.020，最小 ESS = 4426


## 4 · 後驗預測檢查

從後驗採樣一束模型曲線，看它們能不能包住觀測資料——這是「模型和資料一致嗎」的視覺檢驗。

![後驗預測](../figures/04_posterior_predictive.png)

模型束緊貼分箱資料、殘差無明顯結構——後驗預測通過。

## 5 · 限制與誠實的部分

- **`a/R*` 偏高**：我們得到 a/R*≈4.2，高於已發表 3.51。這是 `Rp/R*`–`a/R*` 簡併 + 長曝光摺疊的結果。真實分析會用 **星震學的恆星密度**當 `a/R*` 的強先驗來打破簡併——Kepler-10 的密度量得極準。我們刻意不加，好讓簡併現形（本專案的教學重點）。
- **約化 χ²≈2**：分箱誤差略低估了折疊後的相關雜訊；已加 jitter 項部分吸收，殘差 std≈10 ppm ≈ 噪音水準。
- **固定週期**：在摺疊下 P 視為已知（BLS 值，對上已發表到 ppm 級）；完整分析會同時擬合 P。
- **臨邊昏暗靠先驗**：淺凌日幾乎不約束 LD，結果依賴恆星模型的理論 LD 值。

## 重點

> 凌日參數的強簡併不是缺陷，是物理事實。**貝葉斯 + MCMC 的價值就在誠實呈現它**——
> 一根彎曲的香蕉，而不是一個假裝很確定的點。這也是為什麼天文界的行星論文用 MCMC/nested sampling，不用 Mean-Field VI。